# Document Q&A Chatbot with Gradio

This Colab notebook builds a custom chatbot UI for uploaded PDF documents using **Gradio Blocks**.

### Features included
- PDF upload and processing
- Chat-style question answering over uploaded PDFs
- **Multiple file upload**
- **Answer mode dropdown** (`Concise`, `Detailed`, `Quote-heavy`)
- **Loading / status box**
- **Save chat history to JSON**
- Clear chat button

In [ ]:
!pip -q install gradio pymupdf scikit-learn

In [ ]:
import os
import re
import json
import uuid
import fitz
import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Core retrieval helpers

This app keeps things lightweight and Colab-friendly:
- It extracts text from uploaded PDFs with **PyMuPDF**
- Splits text into chunks
- Uses **TF-IDF + cosine similarity** to retrieve the most relevant chunks
- Returns an answer synthesized from the top matching passages

In [ ]:
APP_STATE = {
    "index": None,
    "chunks": [],
    "sources": [],
    "vectorizer": None,
    "matrix": None,
    "chat_log": []
}


def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text or "").strip()
    return text


def extract_pdf_pages(file_path: str):
    doc = fitz.open(file_path)
    pages = []
    for i in range(len(doc)):
        txt = clean_text(doc[i].get_text("text"))
        if txt:
            pages.append({"page": i + 1, "text": txt})
    return pages


def chunk_text(text: str, chunk_size: int = 900, overlap: int = 150):
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks


def build_index(files):
    all_chunks = []
    all_sources = []

    for file_obj in files:
        file_path = file_obj.name if hasattr(file_obj, "name") else file_obj
        filename = os.path.basename(file_path)
        pages = extract_pdf_pages(file_path)
        for page in pages:
            page_chunks = chunk_text(page["text"])
            for idx, chunk in enumerate(page_chunks):
                all_chunks.append(chunk)
                all_sources.append({
                    "filename": filename,
                    "page": page["page"],
                    "chunk_id": idx
                })

    if not all_chunks:
        raise ValueError("No readable text found in the uploaded PDF(s).")

    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(all_chunks)

    APP_STATE["chunks"] = all_chunks
    APP_STATE["sources"] = all_sources
    APP_STATE["vectorizer"] = vectorizer
    APP_STATE["matrix"] = matrix
    APP_STATE["index"] = True
    APP_STATE["chat_log"] = []

    return f"Processed {len(files)} file(s), {len(all_chunks)} searchable chunk(s)."

In [ ]:
def retrieve_chunks(question: str, top_k: int = 3):
    if APP_STATE["index"] is None:
        return []

    q_vec = APP_STATE["vectorizer"].transform([question])
    sims = cosine_similarity(q_vec, APP_STATE["matrix"])[0]
    ranked = sims.argsort()[::-1][:top_k]

    results = []
    for idx in ranked:
        results.append({
            "score": float(sims[idx]),
            "text": APP_STATE["chunks"][idx],
            "source": APP_STATE["sources"][idx]
        })
    return results


def format_answer(question: str, retrieved, mode: str):
    if not retrieved:
        return "I could not find relevant content in the uploaded document(s)."

    if mode == "Concise":
        best = retrieved[0]
        src = best["source"]
        return (
            f"Best match from **{src['filename']}** page **{src['page']}**:

"
            f"{best['text'][:500]}"
        )

    if mode == "Quote-heavy":
        lines = []
        for item in retrieved:
            src = item["source"]
            lines.append(
                f"- {src['filename']} | page {src['page']}
  "{item['text'][:350]}""
            )
        return "Relevant passages:

" + "

".join(lines)

    # Detailed
    summary_lines = []
    for item in retrieved:
        src = item["source"]
        summary_lines.append(
            f"Source: {src['filename']} page {src['page']}
{item['text'][:420]}"
        )
    return "I searched the uploaded files and found these relevant sections:

" + "

".join(summary_lines)


def answer_question(message, history, mode):
    if APP_STATE["index"] is None:
        bot_reply = "Please upload and process at least one PDF first."
    else:
        retrieved = retrieve_chunks(message, top_k=3)
        bot_reply = format_answer(message, retrieved, mode)

    history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": bot_reply}]
    APP_STATE["chat_log"].append({"question": message, "answer": bot_reply, "mode": mode})
    return history, ""


def clear_chat():
    APP_STATE["chat_log"] = []
    return []


def save_chat_history():
    if not APP_STATE["chat_log"]:
        temp_path = "/tmp/chat_history.json"
        with open(temp_path, "w") as f:
            json.dump([], f, indent=2)
        return temp_path, "Saved an empty chat log."

    filename = f"chat_history_{uuid.uuid4().hex[:8]}.json"
    temp_path = os.path.join("/tmp", filename)
    with open(temp_path, "w") as f:
        json.dump(APP_STATE["chat_log"], f, indent=2)
    return temp_path, f"Saved {len(APP_STATE['chat_log'])} chat turn(s)."

## Build the custom Gradio Blocks interface

This layout includes an extra feature set beyond the tutorial:
- **Multiple file upload**
- **Mode selection dropdown**
- **Save chat history button**
- **Status panel**

In [ ]:
custom_css = """
#title-box {text-align: center; margin-bottom: 8px;}
.gradio-container {max-width: 1200px !important;}
footer {visibility: hidden;}
"""

with gr.Blocks(title="PDF Research Buddy", theme=gr.themes.Soft(), css=custom_css) as demo:
    gr.Markdown(
        """
        <div id='title-box'>
        <h1>📘 PDF Research Buddy</h1>
        <p>Upload one or more PDFs, process them, and ask questions in a chat interface.</p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat History", type="messages", height=480)
            user_input = gr.Textbox(
                label="Your Question",
                placeholder="Ask a question about your uploaded PDF(s)..."
            )
            with gr.Row():
                send_btn = gr.Button("📤 Send", variant="primary")
                clear_btn = gr.Button("🗑️ Clear Chat")

        with gr.Column(scale=1):
            pdf_input = gr.File(
                label="📄 Upload PDF files",
                file_types=[".pdf"],
                file_count="multiple"
            )
            mode_dropdown = gr.Dropdown(
                choices=["Concise", "Detailed", "Quote-heavy"],
                value="Detailed",
                label="Answer Mode"
            )
            process_btn = gr.Button("🔄 Process Document(s)", variant="primary")
            status_box = gr.Textbox(label="Status", interactive=False)
            save_btn = gr.Button("💾 Save Chat History")
            download_file = gr.File(label="Download Chat History")

    process_btn.click(
        fn=build_index,
        inputs=pdf_input,
        outputs=status_box,
        show_progress=True
    )

    send_btn.click(
        fn=answer_question,
        inputs=[user_input, chatbot, mode_dropdown],
        outputs=[chatbot, user_input],
        show_progress=True
    )

    user_input.submit(
        fn=answer_question,
        inputs=[user_input, chatbot, mode_dropdown],
        outputs=[chatbot, user_input],
        show_progress=True
    )

    clear_btn.click(fn=clear_chat, outputs=chatbot)

    save_btn.click(
        fn=save_chat_history,
        inputs=None,
        outputs=[download_file, status_box]
    )

## Launch the app

Run the next cell to open the app in Colab. Set `share=True` so you can submit the public link.

In [ ]:
demo.launch(share=True, debug=True)

## Reflection prompt answers

**Which part felt easiest?** The Gradio layout and button wiring felt easiest because Blocks makes it simple to connect Python functions to visible UI elements.

**Which part felt most challenging?** The most challenging part was keeping the retrieval pipeline lightweight enough for Colab while still making the chatbot feel useful.

**One-sentence app description:** This app is a chat-based PDF assistant that lets users upload one or more documents, process them, and ask questions with selectable answer styles.